# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pingouin
    print('Packages installed.')


# Repeated Measures ANOVA In-Depth

This notebook performs a rigorous Repeated Measures ANOVA using `pingouin`. It includes data generation, assumption checking (Sphericity, Normality), and effect size calculation.

In [ ]:
import pandas as pd
import numpy as np
import pingouin as pg
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style('whitegrid')

## 1. Generate Experimental Data
We simulate a study where 30 subjects undergo a memory test under three conditions: 'Baseline', 'Stress', and 'Relaxation'. We expect Stress to lower scores and Relaxation to improve them.

In [ ]:
np.random.seed(123)
n_subjects = 30

data = {
    'Subject': np.repeat(range(1, n_subjects + 1), 3),
    'Condition': np.tile(['Baseline', 'Stress', 'Relaxation'], n_subjects),
    'Score': np.hstack([
        np.random.normal(70, 10, n_subjects),   # Baseline
        np.random.normal(60, 10, n_subjects),   # Stress
        np.random.normal(78, 10, n_subjects)    # Relaxation
    ])
}
df = pd.DataFrame(data)

print(df.head())

## 2. Visualization
Before testing, we visualize the distributions and means.

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='Condition', y='Score', data=df, palette='Set2')
sns.stripplot(x='Condition', y='Score', data=df, color='black', alpha=0.3)
plt.title('Memory Scores by Condition')
plt.show()

## 3. Assumption Checking
Repeated Measures ANOVA assumes Sphericity (variances of the differences between all combinations of related groups are equal).

In [ ]:
# Mauchly's test for sphericity
sphericity = pg.sphericity(data=df, dv='Score', subject='Subject', within='Condition')
print(sphericity)

## 4. Run Repeated Measures ANOVA
If sphericity is met (p > .05), we use the standard p-value. If violated, we use Greenhouse-Geisser correction.

In [ ]:
aov = pg.rm_anova(data=df, dv='Score', within='Condition', subject='Subject', detailed=True, effsize='np2')
print(aov)

**Interpretation:**
*   **p-unc:** Uncorrected p-value.
*   **np2:** Partial eta-squared (Effect size). 0.01 is small, 0.06 is medium, 0.14 is large.
*   If p < 0.05, there is a significant difference between conditions.

## 5. Post-hoc Tests
If the ANOVA is significant, we need to find out *which* specific conditions differ. We use pairwise t-tests with Bonferroni correction.

In [ ]:
posthoc = pg.pairwise_tests(data=df, dv='Score', within='Condition', subject='Subject', padjust='bonf')
print(posthoc)

## 6. Interaction Plot
Visualizing the change across conditions for each subject can show individual variability.

In [ ]:
plt.figure(figsize=(10, 6))
sns.pointplot(data=df, x='Condition', y='Score', capsize=.1, linestyles='--')
plt.title('Mean Scores with Confidence Intervals')
plt.show()